# Laboratorio 09 — La libreta desde Scala

**Duración: 45 minutos.**

Este laboratorio nace de una pregunta que se hizo en el curso: **¿se puede consumir
Iceberg desde Scala?**

La respuesta corta es que sí. La larga es lo que vas a hacer hoy: correr, desde una
celda de tu propio cuaderno, un programa Scala que abre tu tabla, la lee, le escribe
una fila y te dice dónde viven sus archivos. Sin Spark en ninguna parte.

---

### Lo que cierra este laboratorio

Hasta ahora la tabla estuvo siempre del otro lado de un motor. La escribiste
desde el cuaderno y la leíste desde Hue, en el laboratorio 02 y otra vez en el 07, al
terminar la migración. Hoy la abre un programa que no es ninguno de ellos.

Si eso funciona, la frase *"la tabla no le pertenece a ningún motor"* deja de ser una
promesa del curso y pasa a ser algo que viste correr en tu pantalla.


## Paso 0 — Qué vas a correr, y la libreta de hoy

**En el almacén.** Hasta ahora, cada vez que alguien quiso leer la libreta, la leyó
con los lentes del almacenero: su cuaderno, o Hue. Hoy llega alguien de afuera, con sus
propios lentes, mira el mismo clavo de la pared, descuelga la misma libreta y la lee.
Nadie le prestó nada.

**El hecho técnico.** Iceberg es dos cosas: un formato de archivos, escrito y público,
y una biblioteca que sabe leerlo. **No hay un servidor de Iceberg.** No existe un
proceso al que haya que pedirle permiso, ni una conexión que abrir contra él. Quien
tenga la biblioteca y sepa dónde está el clavo, lee y escribe la tabla.

**Qué es un jar.** Un programa Java empaquetado en un solo archivo, con todo lo que
necesita adentro. Se ejecuta con `java -jar archivo.jar` y no hay nada que instalar. El
de hoy se llama `sin-spark.jar` y ya está en tu ambiente, en `/opt/herramientas/`.

Primero, ponte en tu espacio de trabajo.

In [1]:
%%sql
-- Celda 0.1
USE mi_espacio

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).


26/09/25 18:08:42 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Partimos de cero, por si ya habías corrido este laboratorio antes.

In [2]:
%%sql
-- Celda 0.2
DROP TABLE IF EXISTS contribuyentes_lab09

La tabla de hoy. Los mismos cuatro contribuyentes del padrón, y otra vez
`USING iceberg`.

In [3]:
%%sql
-- Celda 0.3
CREATE TABLE contribuyentes_lab09 (
    rut           STRING,
    razon_social  STRING,
    segmento      STRING
) USING iceberg

Los cuatro contribuyentes. Esta carga deja **una** página en la libreta, y es la
única que va a existir hasta que corras el programa.

In [4]:
%%sql
-- Celda 0.4
INSERT INTO contribuyentes_lab09 VALUES
    ('77746521-K', 'Pehuen Logistica EIRL',    'MICRO'),
    ('77884562-8', 'Araucaria Ferreteria SpA', 'PEQUENA'),
    ('78800840-6', 'Copihue Maquinarias EIRL', 'MEDIANA'),
    ('76171162-8', 'Huemul Alimentos EIRL',    'GRANDE')

## Paso 1 — El código, sin Spark

Antes de correrlo, míralo. Esto es el archivo `SinSpark.scala` **entero**, tal
como está en el repositorio: lo que lees aquí es exactamente lo que corre el jar.

```scala
package cl.sii.iceberg

import java.time.Instant
import java.time.ZoneOffset
import java.time.format.DateTimeFormatter
import java.util.UUID
import java.util.function.{Function => FuncionJava}

import scala.collection.JavaConverters._

import org.apache.hadoop.conf.Configuration
import org.apache.iceberg.PartitionSpec
import org.apache.iceberg.catalog.TableIdentifier
import org.apache.iceberg.data.{GenericRecord, IcebergGenerics, Record}
import org.apache.iceberg.data.parquet.GenericParquetWriter
import org.apache.iceberg.expressions.Expressions
import org.apache.iceberg.hive.HiveCatalog
import org.apache.iceberg.io.CloseableIterable
import org.apache.iceberg.parquet.{Parquet, ParquetValueWriter}
import org.apache.parquet.schema.MessageType

/**
 * La misma tabla, sin Spark en ninguna parte.
 *
 * Este jar no lleva Spark adentro y esa ausencia es el punto: una tabla
 * Iceberg es un directorio de archivos Parquet mas unos manifiestos que dicen
 * cuales cuentan, y cualquier programa que sepa leer ese formato la lee. No
 * hace falta el motor que la escribio, ni ningun servidor que preste el dato.
 *
 * Se lee de arriba abajo, en cinco pasos numerados, y cada cosa se crea en el
 * paso anterior al que la usa. Es lo que el alumno tiene delante en el
 * laboratorio 09: esto es exactamente lo que corre el jar.
 */
object SinSpark {

  def main(argumentos: Array[String]): Unit = {

    // El espacio y la tabla los pone quien lanza el programa, nunca este
    // archivo: asi el mismo programa sirve para cualquier tabla, y ninguna
    // queda amarrada a un nombre escrito aqui.
    val espacio = sys.env.getOrElse("ESPACIO", "")
    val tabla = sys.env.getOrElse("TABLA", "")
    if (espacio.isEmpty || tabla.isEmpty) {
      println("Faltan las variables ESPACIO y TABLA. El programa se lanza asi:")
      println("  ESPACIO=mi_espacio TABLA=contribuyentes_lab09 java -jar sin-spark.jar")
      return
    }
    println(s"Tabla pedida: $espacio.$tabla")

    // 1. Decirle donde esta HDFS. La direccion es la del ambiente.
    val hadoop = new Configuration()
    hadoop.set("fs.defaultFS", "hdfs://namenode:8020")
    hadoop.set("dfs.client.use.datanode.hostname", "true")
    hadoop.set("hive.metastore.uris", "thrift://hive-metastore:9083")
    println()
    println("1. HDFS en hdfs://namenode:8020")

    // 2. Conectarse al catalogo, que es el clavo del que cuelga la libreta.
    val catalogo = new HiveCatalog()
    catalogo.setConf(hadoop)
    catalogo.initialize("laboratorio", Map(
      "uri" -> "thrift://hive-metastore:9083",
      "warehouse" -> "hdfs://namenode:8020/warehouse/iceberg").asJava)
    println("2. Catalogo Hive en thrift://hive-metastore:9083")

    // 3. Pedir la tabla. Esto lee la portada y todavia ningun dato.
    val libreta = catalogo.loadTable(TableIdentifier.of(espacio, tabla))
    val reloj = DateTimeFormatter.ofPattern("yyyy-MM-dd HH:mm:ss.SSS").withZone(ZoneOffset.UTC)
    val paginas = libreta.snapshots().asScala.toSeq.sortBy(_.timestampMillis())
    println()
    println("3. Portada leida. Columnas: " +
      libreta.schema().columns().asScala.map(_.name()).mkString(", "))
    println("   Paginas de la libreta, en hora UTC, igual que .snapshots:")
    paginas.foreach(pagina => println("   %s   %s   %s".format(
      pagina.snapshotId(),
      reloj.format(Instant.ofEpochMilli(pagina.timestampMillis())),
      pagina.operation())))

    // 4. Leer las filas de la pagina vigente y de la primera pagina.
    def mostrar(titulo: String, registros: CloseableIterable[Record]): Unit = {
      println("   " + titulo)
      try {
        val filas = registros.asScala.toSeq.map(registro => "   %-12s %-26s %s".format(
          registro.getField("rut"), registro.getField("razon_social"),
          registro.getField("segmento")))
        filas.sorted.foreach(println)
        println("   %d filas".format(filas.size))
      } finally registros.close()
    }
    println()
    println("4. Las filas, leidas registro a registro y sin motor de consultas.")
    mostrar("Pagina vigente:", IcebergGenerics.read(libreta).build())
    if (paginas.nonEmpty) {
      val primera = paginas.head.snapshotId()
      mostrar(s"Primera pagina, la $primera:",
        IcebergGenerics.read(libreta).useSnapshot(primera).build())
    }

    // 5. Escribir la fila de Litre Transportes: el Parquet, el manifiesto y el commit.
    val rut = "79856201-3"
    val buscado = IcebergGenerics.read(libreta).where(Expressions.equal("rut", rut)).build()
    val yaEsta = try buscado.iterator().hasNext finally buscado.close()
    println()
    if (yaEsta) {
      println(s"5. El contribuyente $rut ya estaba en la tabla. No se escribe de nuevo.")
      println("   Esta celda se puede correr las veces que quieras.")
    } else {
      val registro = GenericRecord.create(libreta.schema())
      registro.setField("rut", rut)
      registro.setField("razon_social", "Litre Transportes SpA")
      registro.setField("segmento", "MICRO")
      val destino = libreta.io().newOutputFile(
        libreta.locationProvider().newDataLocation(s"sin-spark-${UUID.randomUUID()}.parquet"))
      val escritor = Parquet.writeData(destino)
        .schema(libreta.schema())
        .createWriterFunc(new FuncionJava[MessageType, ParquetValueWriter[_]] {
          override def apply(tipo: MessageType): ParquetValueWriter[_] =
            GenericParquetWriter.buildWriter(tipo)
        })
        .withSpec(PartitionSpec.unpartitioned())
        .overwrite()
        .build[Record]()
      try escritor.write(registro) finally escritor.close()
      println(s"5. Parquet escrito: ${destino.location()}")
      libreta.newAppend().appendFile(escritor.toDataFile).commit()
      libreta.refresh()
      println(s"   Lista de manifiestos de esa pagina: ${libreta.currentSnapshot().manifestListLocation()}")
      println(s"   Commit hecho. Pagina nueva ${libreta.currentSnapshot().snapshotId()}.")
    }

    println()
    println("Fin. Ni la lectura ni la escritura pasaron por Spark. No hay Spark en este jar.")
  }
}
```

**Cómo se lee.** De arriba abajo. Cada cosa se crea en el paso anterior al que la
usa, y los cinco números de los comentarios son los cinco movimientos del programa. No
hay ninguno más.

Antes de los cinco, dos líneas que importan: el programa lee `ESPACIO` y `TABLA` de las
variables de entorno. **No trae ningún nombre de tabla escrito adentro**, y por eso la
celda del paso 2 se los pasa. Así el mismo programa sirve para cualquier tabla, y
ninguna queda amarrada a un nombre escrito en el código.

**1. Decirle dónde está HDFS.** Una `Configuration` de Hadoop con la dirección del
almacenamiento. Es la del ambiente, así que va escrita.

**2. Conectarse al catálogo.** El `HiveCatalog` es el clavo de la pared: lo único que
sabe es qué libreta cuelga de él. Se le dice dónde está el Metastore y dónde está el
almacenamiento, y con eso basta. No hay usuario, no hay contraseña, no hay sesión.

**3. Pedir la tabla.** `loadTable` devuelve la tabla: su esquema, sus páginas y dónde
están sus archivos. Fíjate en lo que **no** hizo: todavía no leyó un solo dato. Descolgó
la libreta y miró la portada. Las horas de las páginas salen en UTC, las mismas que vas
a ver después con `.snapshots` desde tu cuaderno.

**4. Leer las filas.** `IcebergGenerics.read(libreta).build()` entrega la tabla como
está ahora, registro a registro. Con `.useSnapshot(primera)`, la misma tabla como estaba
en la página que le indiques: es el `VERSION AS OF` del laboratorio 01, desde Scala y
sin motor de consultas de por medio.

**5. Escribir la fila.** Son tres movimientos, y están a la vista. Se arma el registro,
se escribe **un archivo Parquet nuevo** en el directorio de la tabla, y
`newAppend().appendFile(...).commit()` declara ese archivo como parte de la tabla, que
es lo que crea la página. Si Litre Transportes ya está, el programa lo dice y no escribe
nada: se puede correr las veces que quieras.

Y una cosa que no está. **Ninguna importación dice `org.apache.spark`**: todas empiezan
con `org.apache.iceberg`, `org.apache.hadoop` o `java`. No es un descuido, y al final del
laboratorio lo vas a comprobar contando.

## Paso 2 — Correrlo

Ahora lo ejecutas. Esta celda **no lleva `%%sql`**: no es una consulta, es una orden
del sistema, y por eso empieza con `!`.

Ya viene escrita y no hay que cambiarle nada.

Demora unos segundos. Va a imprimir la tabla que le pediste, después **los cinco pasos
numerados del programa**, en el mismo orden en que los acabas de leer, y el `Fin`. El
paso 5 es donde deja de mirar y escribe.

In [5]:
# Celda 2.1
!ESPACIO=mi_espacio TABLA=contribuyentes_lab09 java -jar /opt/herramientas/sin-spark.jar

Tabla pedida: mi_espacio.contribuyentes_lab09



1. HDFS en hdfs://namenode:8020


2. Catalogo Hive en thrift://hive-metastore:9083



3. Portada leida. Columnas: rut, razon_social, segmento
   Paginas de la libreta, en hora UTC, igual que .snapshots:
   2266374146400284423   2026-09-25 18:08:49.190   append

4. Las filas, leidas registro a registro y sin motor de consultas.


   Pagina vigente:


   76171162-8   Huemul Alimentos EIRL      GRANDE
   77746521-K   Pehuen Logistica EIRL      MICRO
   77884562-8   Araucaria Ferreteria SpA   PEQUENA
   78800840-6   Copihue Maquinarias EIRL   MEDIANA
   4 filas
   Primera pagina, la 2266374146400284423:


   76171162-8   Huemul Alimentos EIRL      GRANDE
   77746521-K   Pehuen Logistica EIRL      MICRO
   77884562-8   Araucaria Ferreteria SpA   PEQUENA
   78800840-6   Copihue Maquinarias EIRL   MEDIANA
   4 filas


5. Parquet escrito: hdfs://namenode:8020/warehouse/iceberg/mi_espacio.db/contribuyentes_lab09/data/sin-spark-a3f48485-af46-43f9-8742-3108d43fc10f.parquet


   Lista de manifiestos de esa pagina: hdfs://namenode:8020/warehouse/iceberg/mi_espacio.db/contribuyentes_lab09/metadata/snap-2188129110206532514-1-9e598506-005e-4c96-a836-77d9a3d12317.avro
   Commit hecho. Pagina nueva 2188129110206532514.

Fin. Ni la lectura ni la escritura pasaron por Spark. No hay Spark en este jar.


## Paso 3 — Comprobar en SQL

El programa dijo que escribió. Compruébalo por tu cuenta, con SQL, que es el terreno
que conoces.

Primero, lo mismo que hiciste en el laboratorio 02 cuando escribiste desde Hue: **pídele
a tu motor que refresque la definición de la tabla**. La fila la escribió otro proceso, y
el cuaderno todavía tiene en memoria la tabla como estaba antes. El dato ya está escrito
desde que el programa dijo «Commit hecho»; el que está atrasado es este motor.

In [6]:
%%sql
-- Celda 3.1
REFRESH TABLE contribuyentes_lab09

Y ahora sí, las filas.

In [7]:
%%sql
-- Celda 3.2
SELECT * FROM contribuyentes_lab09 ORDER BY rut

rut,razon_social,segmento
76171162-8,Huemul Alimentos EIRL,GRANDE
77746521-K,Pehuen Logistica EIRL,MICRO
77884562-8,Araucaria Ferreteria SpA,PEQUENA
78800840-6,Copihue Maquinarias EIRL,MEDIANA
79856201-3,Litre Transportes SpA,MICRO


**Cinco filas**, y la quinta es `Litre Transportes SpA`. Esa fila no la escribió tu
cuaderno ni Hue.

Ahora las páginas. El nombre va **con las tres partes**, como siempre que se consulta
una vista del sistema.

In [8]:
%%sql
-- Celda 3.3
SELECT snapshot_id, committed_at, operation
FROM mi_espacio.contribuyentes_lab09.snapshots
ORDER BY committed_at

snapshot_id,committed_at,operation
2266374146400284423,2026-09-25 18:08:49.190,append
2188129110206532514,2026-09-25 18:08:52.227,append


## Paso 4 — Las dependencias

**Dos páginas.** La primera es tu `INSERT`. La segunda la escribió un programa que no
tiene Spark adentro.

Y si mañana quisieras hacer esto en un proyecto tuyo, esto es todo lo que habría que
declarar:

```scala
"org.apache.iceberg" % "iceberg-core"           % "1.3.0"
"org.apache.iceberg" % "iceberg-data"           % "1.3.0"
"org.apache.iceberg" % "iceberg-parquet"        % "1.3.0"
"org.apache.iceberg" % "iceberg-hive-metastore" % "1.3.0"
"org.apache.hive"    % "hive-metastore"         % "3.1.3"
"org.apache.hadoop"  % "hadoop-client"          % "3.3.6"
```

**Seis líneas**, que son las que declara el módulo `sinSpark` del proyecto de hoy. Las
cuatro primeras son Iceberg. Las dos últimas también hacen falta: el cliente del
Metastore, porque el `HiveCatalog` lo necesita para preguntar qué libreta cuelga del
clavo, y el de Hadoop, porque los archivos están en HDFS.

Con eso un proyecto Scala lee y escribe tablas Iceberg. No hay un servidor que instalar
ni un servicio que levantar, porque no existe.

## Paso 5 — El mismo programa con Spark

Y si lo que quieres es conectar un Spark cualquiera —el de tu máquina, el de un
proceso nocturno, el de la plataforma— al mismo Metastore, esto es lo que se copia y se
pega, desde el `builder` hasta la primera consulta:

```scala
val spark = SparkSession
  .builder()
  .appName("cliente-scala-con-spark")
  .master("local[*]")
  .config("spark.sql.extensions",
          "org.apache.iceberg.spark.extensions.IcebergSparkSessionExtensions")
  .config("spark.sql.catalog.spark_catalog",
          "org.apache.iceberg.spark.SparkSessionCatalog")
  .config("spark.sql.catalog.spark_catalog.type", "hive")
  .config("spark.sql.catalog.spark_catalog.uri", "thrift://hive-metastore:9083")
  .config("spark.sql.catalogImplementation", "hive")
  .config("spark.sql.warehouse.dir", "hdfs://namenode:8020/warehouse/iceberg")
  .config("spark.hadoop.hive.metastore.uris", "thrift://hive-metastore:9083")
  .config("spark.hadoop.hive.metastore.warehouse.dir",
          "hdfs://namenode:8020/warehouse/iceberg")
  .config("spark.hadoop.fs.defaultFS", "hdfs://namenode:8020")
  .config("spark.hadoop.dfs.client.use.datanode.hostname", "true")
  .enableHiveSupport()
  .getOrCreate()

spark.sql("SELECT * FROM mi_espacio.contribuyentes_lab09 ORDER BY rut").show()
```

No lo vamos a ejecutar hoy. Está aquí para que lo tengas escrito cuando lo necesites, y
porque conviene ver que la diferencia entre "leer Iceberg con Spark" y "leer Iceberg sin
Spark" son diez líneas de configuración, no una arquitectura distinta.

## Cierre

Tres procesos distintos leyeron la misma tabla a lo largo del curso:

| Quién | Cuándo |
|---|---|
| Spark, desde tu cuaderno | todos los laboratorios |
| El Spark Thrift Server, que es lo que hay detrás de Hue | laboratorios 02 y 07 |
| Un programa Scala, sin Spark | hoy |

Ninguno le pidió permiso a otro. Ninguno exportó, copió ni sincronizó nada.

**La regla de integración, y es la que te llevas:** cualquier programa que cargue la
biblioteca de Iceberg y sepa dónde está el clavo lee y escribe la misma tabla. Eso es lo
que hace que la plataforma de la DGT pueda cambiar de motor sin mover los datos, y que
una herramienta nueva se conecte sin que nadie tenga que abrirle una puerta.
